<a href="https://www.kaggle.com/code/riteshkumarweb/decisiontree-for-classification?scriptVersionId=317334902" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [20]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/taweilo/loan-approval-classification-data/loan_data.csv


In [21]:
# 🌳 Decision Tree

# Decision Tree is a Supervised ML algorithm
# used for:
# → Classification
# → Regression

# 🧠 It works like a flowchart

# Example:
# Is age > 18 ?
# ├── Yes → Adult
# └── No  → Minor

# 🎯 Goal:
# Split data into pure groups/classes


# 📊 Entropy

# Entropy measures impurity / randomness in data

# High Entropy:
# → Mixed classes ❌

# Low Entropy:
# → Pure classes ✅

# Formula:
# Entropy = - Σ p * log2(p)

# Entropy = 0
# → Completely pure data

# Higher entropy
# → More disorder/randomness


# 📊 Gini Index

# Gini also measures impurity

# Formula:
# Gini = 1 - Σ(p²)

# Gini = 0
# → Pure node

# Lower Gini
# → Better split

# ⚡ Gini is faster than Entropy
# because it avoids logarithms


# 📈 Information Gain

# Information Gain tells:
# "How much uncertainty is reduced after split"

# Formula:
# Information Gain =
# Parent Entropy - Weighted Child Entropy

# 🎯 Decision Tree chooses split
# with Highest Information Gain


# ✅ Advantages of Decision Tree

# ✔ Easy to understand & visualize
# ✔ No feature scaling required
# ✔ Handles non-linear data
# ✔ Works for classification & regression
# ✔ Handles numerical + categorical data


# ❌ Disadvantages of Decision Tree

# ❌ Overfitting problem
# ❌ Sensitive to noise/outliers
# ❌ Can create complex trees
# ❌ Unstable (small data change → different tree)
# ❌ Greedy algorithm (not globally optimal)

In [22]:
df = pd.read_csv('/kaggle/input/datasets/taweilo/loan-approval-classification-data/loan_data.csv')

In [23]:
df

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,27.0,male,Associate,47971.0,6,RENT,15000.0,MEDICAL,15.66,0.31,3.0,645,No,1
44996,37.0,female,Associate,65800.0,17,RENT,9000.0,HOMEIMPROVEMENT,14.07,0.14,11.0,621,No,1
44997,33.0,male,Associate,56942.0,7,RENT,2771.0,DEBTCONSOLIDATION,10.02,0.05,10.0,668,No,1
44998,29.0,male,Bachelor,33164.0,4,RENT,12000.0,EDUCATION,13.23,0.36,6.0,604,No,1


In [24]:
df['loan_status'].value_counts()

loan_status
0    35000
1    10000
Name: count, dtype: int64

In [25]:
df.isnull().sum()

person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64

In [26]:
df['person_education'].value_counts()

person_education
Bachelor       13399
Associate      12028
High School    11972
Master          6980
Doctorate        621
Name: count, dtype: int64

In [27]:
df.sample(10)

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
43040,26.0,male,High School,57335.0,7,RENT,7891.0,MEDICAL,13.38,0.14,5.0,606,No,1
4770,25.0,male,High School,39810.0,0,RENT,5000.0,VENTURE,14.27,0.13,4.0,622,No,1
6626,24.0,male,Bachelor,61966.0,3,RENT,6000.0,MEDICAL,9.63,0.10,3.0,628,Yes,0
13019,23.0,male,Master,102937.0,0,RENT,12000.0,VENTURE,13.16,0.12,4.0,714,Yes,0
1704,24.0,male,High School,26110.0,0,RENT,1800.0,MEDICAL,10.65,0.07,2.0,674,No,0
16697,24.0,male,Master,73039.0,1,RENT,12000.0,EDUCATION,8.90,0.16,3.0,596,Yes,0
25528,32.0,male,Bachelor,102821.0,11,MORTGAGE,7500.0,HOMEIMPROVEMENT,10.78,0.07,10.0,670,Yes,0
16364,24.0,female,Associate,55109.0,2,MORTGAGE,9200.0,HOMEIMPROVEMENT,11.89,0.17,4.0,628,No,0
11589,22.0,female,Associate,85027.0,0,RENT,10000.0,EDUCATION,12.69,0.12,4.0,664,No,0
37321,25.0,female,High School,61330.0,0,RENT,10000.0,HOMEIMPROVEMENT,14.49,0.16,4.0,526,Yes,0


In [28]:
X = df.iloc[:,0:13]
X

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,27.0,male,Associate,47971.0,6,RENT,15000.0,MEDICAL,15.66,0.31,3.0,645,No
44996,37.0,female,Associate,65800.0,17,RENT,9000.0,HOMEIMPROVEMENT,14.07,0.14,11.0,621,No
44997,33.0,male,Associate,56942.0,7,RENT,2771.0,DEBTCONSOLIDATION,10.02,0.05,10.0,668,No
44998,29.0,male,Bachelor,33164.0,4,RENT,12000.0,EDUCATION,13.23,0.36,6.0,604,No


In [29]:
y = df.iloc[:,-1]
y

0        1
1        0
2        1
3        1
4        1
        ..
44995    1
44996    1
44997    1
44998    1
44999    1
Name: loan_status, Length: 45000, dtype: int64

In [30]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
     

In [31]:
X_train

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file
25180,34.0,female,Bachelor,97265.0,11,MORTGAGE,15000.0,PERSONAL,12.73,0.15,9.0,631,No
12555,25.0,male,High School,72953.0,3,RENT,12000.0,VENTURE,11.86,0.16,4.0,659,Yes
29153,41.0,female,Master,322597.0,18,MORTGAGE,24000.0,PERSONAL,10.37,0.07,11.0,683,Yes
23838,27.0,male,Associate,94232.0,4,RENT,9600.0,EDUCATION,17.14,0.10,7.0,641,No
35686,27.0,male,Master,84873.0,7,RENT,7059.0,HOMEIMPROVEMENT,12.97,0.08,3.0,706,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11284,26.0,male,High School,88451.0,6,MORTGAGE,10625.0,EDUCATION,6.03,0.12,4.0,559,Yes
44732,25.0,female,High School,34772.0,3,MORTGAGE,5876.0,HOMEIMPROVEMENT,11.69,0.17,4.0,647,No
38158,33.0,female,Associate,58317.0,9,MORTGAGE,10707.0,VENTURE,10.74,0.18,9.0,652,Yes
860,26.0,male,Master,178602.0,6,RENT,20000.0,DEBTCONSOLIDATION,17.99,0.11,3.0,604,No


In [32]:
# 🌳 Decision Tree Hyperparameters with Example

from sklearn.tree import DecisionTreeClassifier

# 🧠 Create Decision Tree model with hyperparameters

trf4 = DecisionTreeClassifier(

    criterion='gini',      
    # 📊 Split quality measure
    # 'gini' or 'entropy'

    max_depth=5,           
    # 🌲 Maximum depth of tree
    # Prevents overfitting

    min_samples_split=4,   
    # ✂️ Minimum samples needed to split a node

    min_samples_leaf=2,    
    # 🍃 Minimum samples required in a leaf node

    max_leaf_nodes=10,     
    # 🌿 Maximum number of leaf nodes

    max_features='sqrt',   
    # 📦 Number of features considered at each split

    splitter='best',       
    # 🎯 Choose best split
    # Options: 'best', 'random'

    random_state=42        
    # 🔁 Ensures same results every run
)


In [33]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

edu = ['High School','Associate','Bachelor','Master','Doctorate']

preprocessor = ColumnTransformer([

    ('ohe',
     OneHotEncoder(),
     ['person_gender','person_home_ownership','loan_intent','previous_loan_defaults_on_file']),

    ('ordinal',
     OrdinalEncoder(categories=[edu]),
     ['person_education']),

    ('scale',
     StandardScaler(),
     ['person_age','person_income','loan_amnt',
      'loan_int_rate','cb_person_cred_hist_length',
      'credit_score'])

], remainder='passthrough',force_int_remainder_cols=False)

trf3 = DecisionTreeClassifier(max_depth=9)

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', trf3),
    #('model1',trf4)
])

pipe.fit(X_train, y_train)

from sklearn.metrics import accuracy_score,confusion_matrix
y_pred = pipe.predict(X_test)
print('Accuracy',accuracy_score(y_test,y_pred))
print('confusion_matrix\n',confusion_matrix(y_test,y_pred))

Accuracy 0.9184444444444444
confusion_matrix
 [[6793  197]
 [ 537 1473]]
